In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "6"  

proxy = 'http://10.20.38.38:7890'
os.environ['http_proxy'] = proxy
os.environ['https_proxy'] = proxy
import sys
sys.path.append('/home/ldy/Closed_loop_optimizing/model')
import torch
import numpy as np
from PIL import Image
from torchvision import transforms
import torchvision.models as models
import torch.nn as nn
from einops.layers.torch import Rearrange
import math
import importlib
# import util
# importlib.reload(util)
import json
import open_clip
import torch.nn.functional as F
from torch.utils.data import DataLoader
from diffusion_prior import *
# from custom_pipeline import *
import random
from datetime import datetime
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
# from util import (load_model_endocer, preprocess_image, generate_eeg, visualize_images, visualize_top_images, plot_similarity_and_mse_with_dual_axis, get_gteeg, save_eeg, load_thingstestimagedata, get_image_path, save_amx_similarities, save_results, 
# plot_similarity_range, save_value_function_to_txt)

from util import ( get_gteeg, save_eeg, get_image_path)
from ATMS_retrieval import ATMS, get_eeg_features

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


/home/ldy/miniconda3/envs/BCI/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/ldy/miniconda3/envs/BCI/lib/python3.12/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/home/ldy/miniconda3/envs/BCI/lib/python3.12/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/home/ldy/Closed_loop_optimizing/model/eegdatasets_leaveone.py:23: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle mo

In [2]:

def reward_function(eeg_model, image_path, encoder_model_path, yhat, sub, dnn):
    """
    生成与某张图片对应的脑电信号，并与 groundtruth 进行相似度计算
    :param image: 图片特征向量 [1024]
    :param groundtruth_eeg: groundtruth 的特征向量 [1024]
    :return: EEG信号与groundtruth的相似度
    """
    # 模拟生成的 EEG 信号，实际应替换为 EEG 模型输出
    # syn_eeg_folder = f'/mnt/dataset0/kyw/closed-loop/results/syn_eeg'
    eeg_signal = torch.tensor(get_gteeg(image_path, encoder_model_path, dnn, device))
    # syn_eeg_path = save_eeg(eeg_signal, syn_eeg_folder,file_name=f"gt_{timestamp}.npy" )
    
    eeg_feature = get_eeg_features(eeg_model, eeg_signal.unsqueeze(0), device, sub)
    # print(eeg_feature.shape)
    # print(yhat.shape)
    # distance = torch.sqrt(torch.sum((eeg_feature - yhat) ** 2))
    # eeg_feature= eeg_feature.squeeze()
    # yhat = yhat.squeeze()
    # print(eeg_feature.shape)
    # print(yhat.shape)
    # similarity = torch.dot(eeg_feature,yhat)
    similarity = torch.nn.functional.cosine_similarity(eeg_feature, yhat)
    # cos_sim = F.softmax(cos_sim)
    similarity = (similarity + 1) / 2
    
    # print(similarity)
    return similarity.item(),eeg_feature

In [12]:
# image_folder = '/mnt/dataset0/kyw/closed-loop/image_select'

image_folder = '/mnt/dataset0/jiahua/closed-loop/imageset'
# '/mnt/dataset0/kyw/closed-loop/image_select'#'/mnt/dataset0/ldy/NSD_data/pictures/shared1000'
text_list = sorted(os.listdir(image_folder))
syn_eeg_folder = f'/mnt/dataset0/kyw/closed-loop/results/syn_eeg'
# 过滤掉隐藏文件或其他不需要的文件（如果有）
text_list = [file for file in text_list if not file.startswith('.')]
text_list

['00002_antelope',
 '00006_baseball_bat',
 '00010_baton4',
 '00014_bike',
 '00018_bok_choy',
 '00022_bread',
 '00025_buggy',
 '00031_cart',
 '00037_chaps',
 '00042_chopsticks',
 '00048_coffee_bean',
 '00054_creme_brulee',
 '00055_crepe',
 '00058_crow',
 '00061_cupcake',
 '00064_dessert',
 '00072_elephant',
 '00073_espresso',
 '00075_ferry',
 '00083_glove',
 '00086_goose',
 '00090_grenade',
 '00095_highchair',
 '00099_ice_pack',
 '00103_kettle',
 '00107_lampshade',
 '00111_manatee',
 '00114_metal_detector',
 '00118_muff',
 '00122_okra',
 '00123_omelet',
 '00126_orchid',
 '00131_pear',
 '00134_pickax',
 '00135_pie',
 '00138_pocket',
 '00142_possum',
 '00144_pug',
 '00146_purse',
 '00151_robot',
 '00159_scallop',
 '00163_seed',
 '00167_slide',
 '00177_t-shirt',
 '00180_tape_recorder',
 '00181_television',
 '00190_turkey',
 '00193_volleyball',
 '00196_wheat',
 '00200_wok']

In [13]:
image_gt_folder = ['/mnt/dataset0/ldy/datasets/THINGS_MEG/images_set/test_images/00179_taillight/taillight_13s.jpg'
                        ,'/mnt/dataset0/ldy/datasets/THINGS_MEG/images_set/test_images/00171_spoon/spoon_13n.jpg'
                        ,'/mnt/dataset0/ldy/datasets/THINGS_MEG/images_set/test_images/00001_aircraft_carrier/aircraft_carrier_22s.jpg'
                        ,'/mnt/dataset0/ldy/datasets/THINGS_MEG/images_set/test_images/00032_cashew/cashew_15n.jpg'
                        ,'/mnt/dataset0/ldy/datasets/THINGS_MEG/images_set/test_images/00050_cookie/cookie_15s.jpg']
# image_gt_folder = ['/mnt/dataset0/ldy/datasets/THINGS_MEG/images_set/test_images/00014_bike/bike_14s.jpg'
#                         ,'/mnt/dataset0/ldy/datasets/THINGS_MEG/images_set/test_images/00181_television/television_14n.jpg'
#                         , '/mnt/dataset0/ldy/datasets/THINGS_MEG/images_set/test_images/00177_t-shirt/t-shirt_13s.jpg'
#                         ,'/mnt/dataset0/ldy/datasets/THINGS_MEG/images_set/test_images/00135_pie/pie_15s.jpg'
#                         ,'/mnt/dataset0/ldy/datasets/THINGS_MEG/images_set/test_images/00131_pear/pear_13s.jpg']

image_gt_path = image_gt_folder[0]

dir_name = os.path.basename(os.path.dirname(image_gt_path))  # '00014_bike'
gt_category_id = text_list.index(dir_name) 
gt_category_id

ValueError: '00179_taillight' is not in list

In [5]:
features_filename = '/mnt/dataset0/kyw/closed-loop/image50_embeddings.pt'
# features_filename = '/mnt/dataset1/ldy/Workspace/Closed_loop_optimizing/kyw/closed-loop/image_embeddings.pt'
# features_filename = '/mnt/dataset1/ldy/Workspace/Closed_loop_optimizing/kyw/closed-loop/image50_embeddings.pt'
# '/mnt/dataset0/kyw/closed-loop/image_embeddings.pt'
data = torch.load(features_filename, map_location=device)
image_pool = data
image_pool = image_pool.view(-1, 1024)
image_pool = image_pool.detach().cpu()
image_pool.shape

/tmp/ipykernel_1806921/3765131928.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(features_filename, map_location=device)


torch.Size([600, 1024])

In [6]:
model_id = "stabilityai/stable-diffusion-xl-base-1.0"
vae_latents = torch.load(f'/home/ldy/Closed_loop_optimizing/data/vae_latents/{model_id}/600_combined_latents.pt', map_location=device)


/tmp/ipykernel_1806921/3186920770.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  vae_latents = torch.load(f'/home/ldy/Closed_loop_optimizing/data/vae_latents/{model_id}

In [7]:
dnn = 'alexnet' #'alexnet' #,'cornet_s'
sub = 'sub-01'
# encoder_model_path = f'/mnt/dataset1/ldy/Workspace/Closed_loop_optimizing/kyw/closed-loop/EEG-encoding/EEG_encoder/results/{sub}/synthetic_eeg_data/encoding-end_to_end/dnn-{dnn}/modeled_time_points-all/pretrained-True/lr-1e-05__wd-0e+00__bs-064/model_state_dict.pt'
encoder_model_path = f'/mnt/dataset0/kyw/closed-loop/EEG-encoding/EEG_encoder/results/{sub}/synthetic_eeg_data/encoding-end_to_end/dnn-{dnn}/modeled_time_points-all/pretrained-True/lr-1e-05__wd-0e+00__bs-064/model_state_dict.pt'
gt_eeg_folder = f'/mnt/dataset0/kyw/closed-loop/syn_eeg_gt'
# gt_eeg_folder = f'/mnt/dataset1/ldy/Workspace/Closed_loop_optimizing/data/syn_eeg_gt'

In [8]:
# f_encoder =  f"/mnt/dataset1/ldy/Workspace/Closed_loop_optimizing/kyw/closed-loop/sub_model/{sub}/diffusion_alexnet/pretrained_True/gene_gene/ATM_S_reconstruction_scale_0_1000_40.pth"
f_encoder =  f"/mnt/dataset0/kyw/closed-loop/sub_model/{sub}/diffusion_alexnet/pretrained_True/gene_gene/ATM_S_reconstruction_scale_0_1000_40.pth"
checkpoint = torch.load(f_encoder, map_location=device)

eeg_model = ATMS()  # 例如 ATM_S_reconstruction_scale_0_1000 是 EEG 模型
eeg_model.load_state_dict(checkpoint['eeg_model_state_dict'])


/tmp/ipykernel_1806921/590665322.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(f_encoder, map_location=device)


<All keys matched successfully>

In [9]:
seed = 4  # 你可以根据需要更改随机种子
max_iterations = 50
gamma = 0.9
num_images= 10

# os.makedirs(gt_eeg_folder, exist_ok=True)
# gt_eeg_path = os.path.join(gt_eeg_folder, f"gt_{os.path.splitext(os.path.basename(image_gt_path))[0]}.npy")
# print(f"gt_eeg_path {gt_eeg_path}")
# # if os.path.exists(gt_eeg_path):
# try:
#     synthetic_eeg = torch.from_numpy(np.load(gt_eeg_path))
# except:
#     synthetic_eeg = torch.tensor(get_gteeg(image_gt_path, encoder_model_path, dnn, device)) 
# gt_eeg_path = save_eeg(synthetic_eeg, gt_eeg_folder, file_name=f"gt_{os.path.splitext(os.path.basename(image_gt_path))[0]}.npy" )    

# synthetic_eeg.shape

In [10]:
# data['data_x'].shape

In [11]:
len(text_list)

50

In [7]:
import torch.nn.functional as F

def reward_function_clip_embed(embed1, embed2):
    
    
    # Compute cosine similarity (range [-1, 1])
    cosine_sim = F.cosine_similarity(embed1, embed2, dim=1)
    
    # Normalize to [0, 1] range
    normalized_sim = (cosine_sim + 1) / 2
    
    return normalized_sim.item()

In [8]:
import torch.nn.functional as F

def reward_function_vaelatent(vae1, vae2):
    # Flatten the latents if they're not already 1D
    vae1_flat = vae1.view(-1)
    vae2_flat = vae2.view(-1)
    
    # Compute cosine similarity (range [-1, 1])
    cosine_sim = F.cosine_similarity(vae1_flat.unsqueeze(0), vae2_flat.unsqueeze(0), dim=1)
    
    # Normalize to [0, 1] range
    normalized_sim = (cosine_sim + 1) / 2
    
    return normalized_sim.item()

In [14]:
# tar_vae_latent.shape

In [15]:
# data_y_list

In [3]:
features_filename = '/mnt/dataset0/kyw/closed-loop/image50_embeddings.pt'
clip_image_embeds = torch.load(features_filename, map_location=device).view(-1, 1024)

tar_image_embed = torch.load("/home/ldy/Closed_loop_optimizing/data/clip_embed/open_clip/00001_aircraft_carrier_image_embeds.pt", map_location=device)

/tmp/ipykernel_1252412/1694586581.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  clip_image_embeds = torch.load(features_filename, map_location=device).view(-1, 1024)
/

In [4]:
# tar_image_embed.mean(-1)

In [5]:
clip_image_embeds.shape

torch.Size([600, 1024])

In [9]:
# tar_eeg_features = get_eeg_features(eeg_model, synthetic_eeg.unsqueeze(0), device, sub)
# tar_vae_latent = torch.load("/home/ldy/Closed_loop_optimizing/data/vae_latents/stabilityai/stable-diffusion-xl-base-1.0/00177_t-shirt_latents.pt", map_location=device)
# Initialize lists to store the data
data_x_list = []
data_y_list = []

for i, img_embed in enumerate(clip_image_embeds[::12]):
    # image_path = get_image_path(i, 0, text_list)
    similarity = reward_function_clip_embed(img_embed, tar_image_embed)
    # print(f"similarity {similarity}")
    
    # Append the current data to the lists
    data_x_list.append(img_embed)
    data_y_list.append(-similarity*100)

# Convert lists to tensors and create the data dictionary
data = {
    "data_x": torch.stack(data_x_list),  # Stack image embeddings to form a batch
    "data_y": torch.tensor(data_y_list)  # Convert similarities to tensor
}



In [10]:
# Save the data to data.pth
output_dir = f"/home/ldy/Closed_loop_optimizing/data/pseudo_train_data/open_clip/clip_embed_tar"
os.makedirs(output_dir, exist_ok = True)
torch.save(data, os.path.join(output_dir, f"{dir_name}_data_scaling.pth"))

NameError: name 'dir_name' is not defined

In [28]:
data_y_list

[-66.91052913665771,
 -71.3222861289978,
 -66.01232290267944,
 -77.930748462677,
 -66.67226552963257,
 -72.71137833595276,
 -66.60870909690857,
 -67.72497892379761,
 -69.41810846328735,
 -63.53253126144409,
 -70.29716968536377,
 -68.6979591846466,
 -67.62412786483765,
 -67.50473976135254,
 -64.2970621585846,
 -63.102567195892334,
 -66.21546745300293,
 -68.88190507888794,
 -68.5457170009613,
 -59.576237201690674,
 -69.17233467102051,
 -70.99754810333252,
 -70.69342732429504,
 -62.98251152038574,
 -71.37568593025208,
 -65.52839279174805,
 -64.62202668190002,
 -69.70642805099487,
 -63.90882730484009,
 -66.80094003677368,
 -67.7227795124054,
 -65.40484428405762,
 -70.91583013534546,
 -72.12888598442078,
 -71.42237424850464,
 -68.37502717971802,
 -66.92390441894531,
 -71.3894248008728,
 -63.14911246299744,
 -67.78866052627563,
 -65.55335521697998,
 -69.0251111984253,
 -70.78341245651245,
 -64.43396806716919,
 -72.96488285064697,
 -74.14207458496094,
 -68.73642206192017,
 -67.45108366012573,

In [29]:
data_y_list.index(min(data_y_list))

3

In [ ]:
# tar_eeg_features = get_eeg_features(eeg_model, synthetic_eeg.unsqueeze(0), device, sub)

# # Initialize lists to store the data
# data_x_list = []
# data_y_list = []

# for i, img_embed in enumerate(vae_latents):
#     image_path = get_image_path(i//12, i%12, text_list)
#     similarity, choose_eeg_feature = reward_function(eeg_model, image_path, encoder_model_path, tar_eeg_features, sub, dnn)
#     # print(f"similarity {similarity}")
    
#     # Append the current data to the lists
#     data_x_list.append(img_embed)
#     data_y_list.append(-similarity)

# # Convert lists to tensors and create the data dictionary
# data = {
#     "data_x": torch.stack(data_x_list),  # Stack image embeddings to form a batch
#     "data_y": torch.tensor(data_y_list)  # Convert similarities to tensor
# }

# # Save the data to data.pth
# torch.save(data, "/home/ldy/Closed_loop_optimizing/data/pseudo_train_data/600_data.pth")